In [ ]:
from dotenv import load_dotenv
from crewai import LLM
from markdown_it.rules_core import replacements

# 1. Load your NEW API key
load_dotenv()

# 2. Use the 2026 Stable Model String
# Note the "gemini/" prefix and the "2.5" version
llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.1
)

Here we provide the Original Email,Task and the BaseTool to to b used expand the internal private Jargons used in a company Email.

In [ ]:
original_email="""
           looping in priya.TAS and PRX updates are in the deck.ETA for SDS integration is Friday.
           Lets sync up tomorrow if SYNCBOT allows:). Pink me if any blockers.
           """

#Creating a Tool to replace all the Jargons

from crewai.tools import BaseTool
from typing import Optional

class ReplaceJargonsTool(BaseTool):
    name: str = "Replace Jargons"
    description: str = "Useful for replacing internal jargons or project codenames with clear, specific terms."

    def _run(self, email: str) -> str:
        """Analyzes an email string and suggests replacements for detected jargons."""
        replacements = {
            "PRX": "Project Pheonix (internal AI revamp project)",
            "TAS": "technical architecture stack",
            "DBX": "client database cluster",
            "SDS": "Smart Data syncer",
            "SYNCBOT": "internal standup assistant bot",
            "WIP": "Work in Progress",
            "POC": "Proof of Concept",
            "ping": "reach out"
        }

        suggestions = []
        email_lower = email.lower()

        for jargon, replacement in replacements.items():
            if jargon.lower() in email_lower:
                suggestions.append(f"Consider replacing '{jargon}' with '{replacement}'")

        return "\n".join(suggestions) if suggestions else "No jargons or internal abbreviations detected."

jt=ReplaceJargonsTool()
jt.run(original_email)



In [ ]:
from crewai import Agent,Task,Crew
email_assistant=Agent(
    role="Email Assistant Agent",
    goal="Improve emails and make them sound professional and clear",
    backstory="A highly experienced communication expert skilled in professional email writing",
    tools=[jt],
    verbose=True,
    llm=llm
)

original_email="""
           looping in priya.TAS and PRX updates are in the deck.ETA for SDS integration is Friday.
           Lets sync up tomorrow if SYNCBOT allows:). Pink me if any blockers.
           """
email_task=Task(

     description=f"""Take the following rough email and rewrite into  a professional polished
     version.
     Expand abbervations:
     '''{original_email}'''""",
    agent=email_assistant,
    expected_output="A Professional written email with proper formatting and content.",
)

crew=Crew(
     agents=[email_assistant],
     tasks=[email_task],
    verbose=True,

)

result = crew.kickoff()
print(result)
